In [377]:
import pandas as pd
import pickle
import numpy as np

Compare the weighted e, s, g, esg scores of the otpimised portfolio vs the decarbonised portfolio:

In [378]:
with open("Data/optimal_portfolios_by_te.pkl", "rb") as f:
    optimal_portfolios_by_te = pickle.load(f)

In [379]:
e_score = pd.read_excel("Data/esg_score_comp_1222.xlsm", sheet_name="E", skiprows=4).iloc[4]
s_score = pd.read_excel("Data/esg_score_comp_1222.xlsm", sheet_name="S", skiprows=4).iloc[4]
g_score = pd.read_excel("Data/esg_score_comp_1222.xlsm", sheet_name="G", skiprows=4).iloc[4]
esg_score = pd.read_excel("Data/esg_score_comp_1222.xlsm", sheet_name="ESG", skiprows=4).iloc[261]

In [382]:
type_lseg = pd.read_excel("Data/symbol_comp_1222.xlsm", sheet_name="SYMBOL", dtype=str).iloc[0].values[1:]
symbol_lseg = pd.read_excel("Data/symbol_comp_1222.xlsm", sheet_name="SYMBOL", dtype=str, header=2)
symbol_lseg = symbol_lseg.iloc[:1].transpose().reset_index().rename(columns= {"index": "NAME", 0: "SYMBOL"}).iloc[1:]
symbol_lseg['TYPE'] = type_lseg

In [383]:
symbol_lseg.loc[symbol_lseg['SYMBOL'] == 'BRK.A', 'SYMBOL'] = 'BRK-B'

In [384]:
symbol_lseg

,NAME,SYMBOL,TYPE
1,AMAZON.COM,AMZN,891399
2,ABBOTT LABORATORIES,ABT,916328
3,AES,AES,545101
4,INTERNATIONAL BUS.MCHS.,IBM,906187
5,ADVANCED MICRO DEVICES,AMD,936365
...,...,...,...
499,WEC ENERGY GROUP,WEC,902335
500,MONSTER BEVERAGE,MNST,512785
501,LINDE (NYS),LIN,9373MH
502,SBA COMMS.,SBAC,699786


In [385]:
e_score = e_score.reset_index().iloc[1:].rename(columns={'index':'TYPE', 4:'E_score'})
s_score = s_score.reset_index().iloc[1:].rename(columns={'index':'TYPE', 4:'S_score'})
g_score = g_score.reset_index().iloc[1:].rename(columns={'index':'TYPE', 4:'G_score'})
esg_score = esg_score.reset_index().iloc[1:].rename(columns={'index':'TYPE', 261:'ESG_score'})

In [386]:
e_score.TYPE = e_score.TYPE.apply(lambda x: x.split("(ENSCORE)")[0])
s_score.TYPE = s_score.TYPE.apply(lambda x: x.split("(SOSCORE)")[0])
g_score.TYPE = g_score.TYPE.apply(lambda x: x.split("(CGSCORE)")[0])
esg_score.TYPE = esg_score.TYPE.apply(lambda x: x.split("(TRESGS)")[0])

In [387]:
e_score = pd.merge(e_score, symbol_lseg, how = 'left', on = 'TYPE')
s_score = pd.merge(s_score, symbol_lseg, how = 'left', on = 'TYPE')
g_score = pd.merge(g_score, symbol_lseg, how = 'left', on = 'TYPE')
esg_score = pd.merge(esg_score, symbol_lseg, how = 'left', on = 'TYPE')

In [388]:
symbol_lseg.loc[symbol_lseg['NAME'] == 'NRG ENERGY']

,NAME,SYMBOL,TYPE
80,NRG ENERGY,NRG,28179N


In [389]:
esg_score.loc[esg_score['SYMBOL'].isna()]

,TYPE,ESG_score,NAME,SYMBOL
449,Unnamed: 450,NaN,NaN,NaN


In [390]:
esg_score.loc[esg_score['SYMBOL'].isna()]

,TYPE,ESG_score,NAME,SYMBOL
449,Unnamed: 450,NaN,NaN,NaN


In [391]:
esg_score.head()

,TYPE,ESG_score,NAME,SYMBOL
0,891399,79.48,AMAZON.COM,AMZN
1,916328,80.5,ABBOTT LABORATORIES,ABT
2,545101,74.68,AES,AES
3,906187,73.68,INTERNATIONAL BUS.MCHS.,IBM
4,936365,67.96,ADVANCED MICRO DEVICES,AMD


In [392]:
dict_te_2_percent = optimal_portfolios_by_te[0.02]

*to do: prova a riscaricare PARA

In [401]:
def compute_portfolio_sust_score(sector_dict, sust_df, sust_score):
    results = {}

    sust_map = sust_df.set_index("SYMBOL")[sust_score].to_dict()

    for sector, data in sector_dict.items():
        stock_labels = data["stock_labels"]
        w_b = data["w_b_vec"]
        w_opt = data["w_opt"]

        # Align sust scores
        sust_scores = np.array([sust_map.get(sym, np.nan) for sym in stock_labels])

        # Identify missing tickers
        missing = [sym for sym, score in zip(stock_labels, sust_scores) if np.isnan(score)]
        if missing:
            print(f"⚠️ {sector}: Missing {sust_score} scores for {missing}")

        # Mask valid scores
        mask = ~np.isnan(sust_scores)
        if mask.sum() > 0:
            sust_b = np.dot(w_b[mask], sust_scores[mask]) / w_b[mask].sum()
            sust_opt = np.dot(w_opt[mask], sust_scores[mask]) / w_opt[mask].sum()
          
            assert round(w_b[mask].sum(),2) == 1.0
            assert round(w_opt[mask].sum(),2) == 1.0
        else:
            sust_b, sust_opt = np.nan, np.nan

        results[sector] = {
            f"{sust_score}_original": sust_b,
            f"{sust_score}_optimized": sust_opt,
            "Missing tickers": missing
        }

    return pd.DataFrame(results).T


for sust_df, sust_score in [(esg_score, "ESG_score"), (e_score, "E_score"), (s_score, "S_score"), (g_score, "G_score")]:
    print(compute_portfolio_sust_score(dict_te_2_percent, sust_df, sust_score))
    print()

⚠️ Communication Services: Missing ESG_score scores for ['PARA']
                       ESG_score_original ESG_score_optimized Missing tickers
Consumer Discretionary          69.906132           65.676489              []
Health Care                      75.77177           72.985169              []
Utilities                       69.186304           70.923111              []
Information Technology          77.645036           75.822665              []
Real Estate                      71.30719           67.494901              []
Materials                       75.726529           73.848314              []
Industrials                     70.340152           68.414729              []
Financials                      65.061275           67.525448              []
Energy                          71.489517           70.937451              []
Communication Services          69.522224           69.474036          [PARA]
Consumer Staples                73.537173           73.536978              []

ESG scores piu' bassi nel portafoglio ottimizzato eccetto per il settore Financials